# 6-dars: neyron tarmoq — xato orqaga qanday qaytadi

**Hikoya.** Sizning uyingiz ikki qavatli. Zinaning pastida ham, tepasida ham bittadan tugma bor, ikkalasi bitta lampochkani boshqaradi. Qoida shunday:
- ikkala tugma **bir xil** holatda (ikkalasi o’chiq yoki ikkalasi yoqilgan) — lampochka **o’chiq**;
- tugmalar **har xil** holatda — lampochka **yonadi**.

5-darsda kompyuter «chiziq» topishni o’rgandi. Savol: kompyuter shu **lampochka qoidasini** ham o’rgana oladimi? Bugun ko’ramiz: **bitta chiziq bunga yetmaydi**, shuning uchun **neyron tarmoq** kerak.

**Bugun siz:**
- **bitta neyron** nima ekanini — oddiy «qaror qiluvchi» ekanini ko’rasiz;
- nega bitta neyron lampochka masalasini yecha olmasligini ko’rasiz va **ikki qatlamli tarmoqni qo’lda yig’asiz**;
- tarmoq o’rganayotganda **xato orqaga qanday qaytishini** (backpropagation) tushunasiz;
- o’zingizning **mini-autograd** dasturingizni yozib, tarmoqni o’qitasiz.

**Asosiy fikr:** neyron tarmoq — oddiy «ball hisoblagichlar» zanjiri. Zanjirni o’qitish uchun xatoning har bir bo’g’inga **qancha ta’sir qilganini** bilish kerak. Buni kompyuter xatoni zanjir bo’ylab **orqaga yuborib** topadi.

**Notebook qanday ishlaydi:** matn kataklari — tushuntirish; kod kataklari ▶ (yoki Shift + Enter) bilan ishga tushadi. Kataklarni yuqoridan pastga, tartib bilan ishga tushiring.

⚠️ Avval: **Copy to Drive** (yoki **File → Save a copy in Drive**).

## 0. Kutubxonalar

Bugun yangi kutubxona kerak emas. Faqat Python’ning tayyor `math` (matematika), `random` (tasodifiy son) va `matplotlib` (grafik) kutubxonalari. Tarmoqni **o’zimiz noldan** yozamiz.

In [ ]:
import math
import random
import matplotlib.pyplot as plt

## 1. Bitta neyron — kichik «qaror qiluvchi»

Neyron nima ekanini zinadagi lampochkadan oldin **oddiy qaror** bilan ko’ramiz: **«Bugun futbol o’ynaymizmi?»**

Qaror 3 savolga bog’liq (javob: 1 — ha, 0 — yo’q):
- ob-havo yaxshimi?
- do’stlar boryaptimi?
- uy ishi bormi?

Har savolning **muhimligi** boshqacha. Biz uni songa aylantiramiz va **og’irlik** deymiz:
- ob-havo: **+2** (yaxshi bo’lsa borgingiz keladi);
- do’stlar: **+3** (eng muhim);
- uy ishi: **−4** (borsangiz, minus: borishingiz qiyin).

Ular yig’indisiga ozgina «dangasalik» ham qo’shiladi: **b = −2**. Yig’indi **musbat** bo’lsa — boramiz, **manfiy** bo’lsa — qolamiz.

In [ ]:
def qaror(ob_havo, doslar, uy_ishi):
    ball = 2 * ob_havo + 3 * doslar - 4 * uy_ishi - 2     # har javob o'z og'irligiga ko'paytirilib qo'shiladi
    return ball

print(qaror(1, 1, 0))     # yaxshi ob-havo, do'stlar bor, uy ishi yo'q
print(qaror(1, 1, 1))     # hammasi bor, lekin uy ishi ham bor
print(qaror(0, 0, 0))     # hech narsa yo'q

Natija: **3** (boramiz!), **−1** (uy ishi hamma narsani buzdi), **−2** (qolamiz).

Mana, **neyron** shu: **javoblarni og’irliklariga ko’paytirib qo’shadi va `b` ni qo’shadi**. 5-darsdagi `k × soat + b` ni eslang: u ham aynan shu. Faqat o’sha yerda javob bitta (soat), bu yerda uchta. `k` ning ko’plik shakli — **og’irlik** (inglizcha *weight*, `w`).

> neyron = w₁·x₁ + w₂·x₂ + w₃·x₃ + b

**Sinab ko’ring.** Ob-havo yaxshi, do’stlar **yo’q**, uy ishi **bor**. Boramizmi? Avval taxmin qiling, keyin `qaror` bilan tekshiring.

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

Endi buni **har qanday** javoblar va og’irliklar uchun ishlaydigan funksiyaga aylantiramiz. Javoblar ro’yxati `x`, og’irliklar ro’yxati `w`.

**Mustaqil ish 1.** `neyron(x, w, b)` funksiyasini yozing: har javobni o’z og’irligiga ko’paytirib qo’shing va `b` ni qo’shing.

In [ ]:
# TODO: neyron funksiyasini to'ldiring
def neyron(x, w, b):
    jami = 0
    for xi, wi in zip(x, w):        # zip: javob va uning og'irligini juft-juft oladi
        jami = jami                 # TODO: jami ga xi * wi ni qo'shing
    return jami + b

natija1 = neyron([1, 1, 0], [2, 3, -4], -2)
assert natija1 == 3, f"neyron([1, 1, 0], [2, 3, -4], -2) = {natija1}, lekin 3 chiqishi kerak. jami = jami + xi * wi yozdingizmi?"
print(natija1)              # 3 chiqishi kerak

## 2. Aktivatsiya: balldan «signal» yasash

Neyron ball hisobladi, masalan, 3 yoki −1. Keyingi neyronga **shu ballning o’zini** emas, **«signal»**ni beramiz. Signal yasaydigan qoida **aktivatsiya** deyiladi. Ikkita keng tarqalgani:

- **ReLU**: ball manfiy bo’lsa — **0** («jim»), musbat bo’lsa — o’zi. Kodda: `max(0, ball)`.
- **tanh**: ballni **−1 dan 1 gacha** siqadi. Juda katta ball — deyarli 1, juda kichik — deyarli −1. Yumshoq «ha/yo’q».

In [ ]:
def relu(ball):
    return max(0, ball)                      # manfiyni 0 qiladi

print("relu:", relu(3), relu(-1), relu(-2))      # 3, 0, 0
print("tanh:", [round(math.tanh(b), 2) for b in range(-3, 4)])    # -3 dan 3 gacha

tanh natijasi: −1.0, −0.96, −0.76, 0.0, 0.76, 0.96, 1.0. Ya’ni ball nolga yaqin bo’lsa, tanh o’zgarishni sezadi; ball juda katta bo’lsa — «to’yingan», deyarli 1. Ikkala aktivatsiyani chizib ko’ramiz.

In [ ]:
ballar = [i / 10 for i in range(-30, 31)]          # -3.0, -2.9, ... 3.0
plt.plot(ballar, [relu(b) for b in ballar], label="ReLU")
plt.plot(ballar, [math.tanh(b) for b in ballar], label="tanh")
plt.xlabel("ball"); plt.ylabel("signal"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Ikki aktivatsiya")
plt.show()

Ikkalasi ham **to’g’ri chiziq emas** (ReLU burilgan, tanh egri). Aynan shu «to’g’ri emaslik» keyingi bo’limda hal qiluvchi bo’ladi.

Nega «ha/yo’q» (0 yoki 1) o’rniga silliq aktivatsiya? 5-darsni eslang: o’rganish **qiyalikka** tayanadi. «Ha/yo’q» qoidasi tekis, uning qiyaligi har joyda 0: qaysi tomonga yurishni bilib bo’lmaydi. Silliq aktivatsiyada qiyalik bor.

## 3. Lampochka masalasi: nega bitta neyron yetmaydi

Zinadagi lampochkani **jadvalga** yozamiz. `x1` — pastdagi tugma, `x2` — tepadagi (1 — yoqilgan). `y` — lampochka yonadimi:

| x1 | x2 | y (yonadimi) |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Bu masala **XOR** («faqat bittasi») deb ataladi: lampochka faqat **bitta** tugma yoqilganda yonadi. To’rt nuqtani chizamiz: **qizil** — lampochka yonadi, **ko’k** — o’chiq.

In [ ]:
X = [[0, 0], [0, 1], [1, 0], [1, 1]]     # tugmalar holati
Y = [0, 1, 1, 0]                          # lampochka yonadimi

for (x1, x2), y in zip(X, Y):
    plt.scatter(x1, x2, s=300, color="red" if y == 1 else "blue")
plt.xlabel("x1 (pastdagi tugma)"); plt.ylabel("x2 (tepadagi tugma)")
plt.title("Qizil — yonadi, ko'k — o'chiq")
plt.xlim(-0.5, 1.5); plt.ylim(-0.5, 1.5)
plt.show()

Qizil nuqtalar **qarama-qarshi burchaklarda**. Bitta to’g’ri chiziq bilan qizillarni ko’klardan **ajrata olasizmi**? Qog’ozga chizib ko’ring: bo’lmaydi.

Bitta neyron aynan shunday **bitta to’g’ri chiziq** o’tkazadi (`w₁·x1 + w₂·x2 + b` ning ishorasi). Kompyuter hamma variantni sinab ko’rsin: og’irliklar va `b` ni **−4 dan 4 gacha** (0.5 qadam bilan) 4913 ta kombinatsiyada tekshiramiz va eng yaxshisi nechta nuqtani to’g’ri topishini ko’ramiz.

In [ ]:
qiymatlar = [i / 2 for i in range(-8, 9)]       # -4, -3.5, ... 4
eng_yaxshi = 0

for w1 in qiymatlar:
    for w2 in qiymatlar:
        for b in qiymatlar:
            togri = 0
            for (x1, x2), y in zip(X, Y):
                javob = 1 if w1 * x1 + w2 * x2 + b > 0 else 0    # ball musbat bo'lsa "yonadi"
                togri += (javob == y)
            eng_yaxshi = max(eng_yaxshi, togri)

print("eng yaxshi natija:", eng_yaxshi, "/ 4")

**3 / 4**. 4913 ta kombinatsiyaning **birortasi ham** to’rt nuqtani to’g’ri topa olmaydi. Bitta neyron lampochka qoidasini **o’rganolmaydi**.

**Yechim: ikkinchi qatlam.** Neyronlarni **ikki qatorga** qo’yamiz. Birinchi qatordagi neyronlar javoblarni sanaydi, ikkinchi qatordagi neyron ularni birlashtiradi. Mana **qo’lda yig’ilgan** tarmoq (ReLU bilan):

- **h1 = relu(x1 + x2)**: «nechta tugma yoqilgan?» (0, 1 yoki 2);
- **h2 = relu(x1 + x2 − 1)**: «ikkalasi ham yoqilganmi?» (faqat 2 bo’lsa 1, aks holda 0);
- **chiqish = h1 − 2·h2**: tugmalar sonidan **ikkalasi yoqilgan holat uchun jarima** ayiramiz.

In [ ]:
def xor_tarmoq(x1, x2):
    h1 = relu(x1 + x2)              # nechta tugma yoqilgan
    h2 = relu(x1 + x2 - 1)          # ikkalasi ham yoqilganmi
    return h1 - 2 * h2              # ikkalasi bo'lsa jarima

print("x1 x2 | h1 h2 | chiqish")
for x1, x2 in X:
    h1 = relu(x1 + x2); h2 = relu(x1 + x2 - 1)
    print(f" {x1}  {x2} |  {h1}  {h2} |   {xor_tarmoq(x1, x2)}")

Chiqish ustuni: **0, 1, 1, 0**, aynan lampochka jadvali! Ikki qatlamli tarmoq muammoni hal qildi. Muhim: bu tarmoq **chiziq emas**. Nega? Agar aktivatsiyani olib tashlasak nima bo’ladi?

In [ ]:
print("x1 x2 | aktivatsiyasiz chiqish")
for x1, x2 in X:
    h1 = x1 + x2                    # relu yo'q
    h2 = x1 + x2 - 1                # relu yo'q
    print(f" {x1}  {x2} |   {h1 - 2 * h2}")

Aktivatsiyasiz chiqish: **2, 1, 1, 0**. Bu 0, 1, 1, 0 emas! Matematikasi: `h1 − 2·h2 = x1 + x2 − 2(x1 + x2 − 1) = 2 − x1 − x2`, ya’ni yana **bitta to’g’ri chiziq**. Aktivatsiyasiz qancha qatlam qo’ysangiz ham, hammasi bitta chiziqqa qo’shilib ketadi. **Aktivatsiya (burilish) tarmoqqa chiziq bo’lmagan narsalarni o’rganish imkonini beradi.**

**Sinab ko’ring.** `xor_tarmoq(1, 1)` va `xor_tarmoq(1, 0)` ni chaqiring. Avval taxmin qiling: nima chiqadi?

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

**Mustaqil ish 2.** Tarmoqni o’zingiz yig’ing: `h2` qatori yetishmayapti. Eslatma: «ikkalasi ham yoqilganmi?» = `relu(x1 + x2 - 1)`.

In [ ]:
# TODO: h2 qatorini to'ldiring
def xor_tarmoq2(x1, x2):
    h1 = relu(x1 + x2)
    h2 = 0                          # TODO: relu(x1 + x2 - 1)
    return h1 - 2 * h2

natija2 = [xor_tarmoq2(x1, x2) for x1, x2 in X]
assert natija2 == [0, 1, 1, 0], f"natija {natija2}, lekin [0, 1, 1, 0] chiqishi kerak. h2 = relu(x1 + x2 - 1) yozdingizmi?"
print(natija2)                  # [0, 1, 1, 0] chiqishi kerak

## 4. Tarmoq o’zi o’rganishi kerak

Qo’lda og’irliklarni topdik (1, −2, 1…), chunki masala kichik. Haqiqiy tarmoqda **millionlab** og’irlik bor (Face ID), yoki **yuz milliardlab** (ChatGPT). Ularni qo’lda tanlab bo’lmaydi. 5-darsdagi usul kerak: **loss** va **qiyalik**:

1. loss — tarmoq qanchalik adashdi;
2. har og’irlik uchun qiyalik: **«bu og’irlikni ozgina o’zgartirsam, loss qanday o’zgaradi?»**;
3. har og’irlikni qiyalikka teskari tomonga suramiz: `w = w - lr * qiyalik`.

Bu 5-dars tsikli. Yangi qiyinchilik: tarmoqda **juda ko’p og’irlik** bor, har birining qiyaligini topish kerak. 5-darsdagi «kichik qadam» usuli bilan har og’irlik uchun **butun tarmoqni qayta hisoblash** kerak: millionlab og’irlik — millionlab marta. Sekin!

Aqlliroq yo’l bor: **backpropagation** (orqaga tarqalish). Uni **kichik misolda** ko’ramiz.

### Hisoblash grafi

Bitta kichik formula: **L = (a · b + c) · f**. Uni **kichik amallarga** bo’lamiz, har amal — bitta qadam:
- `d = a · b`
- `e = d + c`
- `L = e · f`

Sonlar: a = 2, b = −3, c = 10, f = −2.

In [ ]:
a, b, c, f = 2.0, -3.0, 10.0, -2.0

d = a * b          # 1-qadam
e = d + c          # 2-qadam
L = e * f          # 3-qadam
print("d =", d, " e =", e, " L =", L)       # d = -6, e = 4, L = -8

Bu — **hisoblash grafi**: sonlar → amallar → natija. Savol: **`a` ni ozgina o’zgartirsam, L qanday o’zgaradi?** Va xuddi shunday `b`, `c`, `f` uchun. 5-darsdagi usul («kichik qadam») bilan topamiz:

In [ ]:
def L_hisobla(a, b, c, f):
    return (a * b + c) * f

h = 0.001                                           # kichik qadam
L0 = L_hisobla(a, b, c, f)                          # hozirgi L
qiy_a = (L_hisobla(a + h, b, c, f) - L0) / h        # a ni surdik
qiy_b = (L_hisobla(a, b + h, c, f) - L0) / h
qiy_c = (L_hisobla(a, b, c + h, f) - L0) / h
qiy_f = (L_hisobla(a, b, c, f + h) - L0) / h

print("a:", round(qiy_a, 1), " b:", round(qiy_b, 1), " c:", round(qiy_c, 1), " f:", round(qiy_f, 1))   # 6.0 -4.0 -2.0 4.0

Natija: **a: 6, b: −4, c: −2, f: 4**. Bu 4 ta qiyalik uchun biz funksiyani **5 marta** hisobladik. 17 og’irlikli tarmoqda 18 marta; millionlab og’irlikda — millionlab marta.

### Ta’sir zanjiri: orqaga qarab

Aqlli yo’l: **oxiridan boshlab**, ta’sirni **orqaga uzatamiz**. Misol: lampochkadagi zanjir. Siz tugmani bosdingiz, sim orqali elektr o’tdi, lampochka yondi. **Lampochka yonmasa, qaysi bo’g’in aybdor?** Oxiridan boshlab tekshirasiz.

Bizda ham: **L** dan boshlaymiz, uning o’zi uchun ta’sir **1** (L ni 1 ga oshirsam, L 1 ga oshadi). Keyin orqaga, har amal o’z qoidasi bilan:

- **Ko’paytirish** (`L = e · f`): `e` ni 1 ga oshirsam, L **f marta** oshadi. Ya’ni `e` ning ta’siri = `f × (L ning ta’siri)`. `f` ning ta’siri = `e × (L ning ta’siri)`.
- **Qo’shish** (`e = d + c`): `d` ni 1 ga oshirsam, `e` ham 1 ga oshadi. Ta’sir **o’zgarishsiz uzatiladi**.

**Zanjir qoidasi** shu: bo’g’inlar ta’sirini **ketma-ket ko’paytirasiz**. Masalan, `a` ni 1 ga oshirsam → `d` **b** marta o’zgaradi → `e` xuddi shuncha → L esa `f` marta. Jami: `b × f`.

In [ ]:
L_ta = 1.0                  # L ning o'z ta'siri

e_ta = f * L_ta             # L = e * f  ->  e o'zgarsa, L f marta o'zgaradi
f_ta = e * L_ta             # f o'zgarsa, L e marta o'zgaradi

d_ta = 1.0 * e_ta           # e = d + c  ->  ta'sir o'zgarishsiz uzatiladi
c_ta = 1.0 * e_ta

a_ta = b * d_ta             # d = a * b  ->  a o'zgarsa, d b marta o'zgaradi
b_ta = a * d_ta

print("a:", a_ta, " b:", b_ta, " c:", c_ta, " f:", f_ta)     # 6.0 -4.0 -2.0 4.0

**Aynan bir xil javob** (6, −4, −2, 4), lekin **bitta orqaga o’tish** bilan: oldinga bir marta (L ni hisobladik), orqaga bir marta (hamma ta’sirni topdik). Og’irlik soni 1 milliard bo’lsa ham, orqaga o’tish **bitta**. **Shu — backpropagation**: xatoning ta’siri zanjir bo’ylab orqaga tarqaladi.

Bu qiyaliklar odatda **grad** (gradient — 5-darsdagi «qiyalik») deb ataladi.

**Mustaqil ish 3.** Orqaga o’tishning ikki qatori yetishmayapti: `a_ta` va `b_ta`. Qoida: `d = a · b`, demak `a` ning ta’siri = **b × d_ta**, `b` ning ta’siri = **a × d_ta**.

In [ ]:
# TODO: a_ta2 va b_ta2 ni hisoblang
L_ta2 = 1.0
e_ta2 = f * L_ta2
f_ta2 = e * L_ta2
d_ta2 = 1.0 * e_ta2
c_ta2 = 1.0 * e_ta2

a_ta2 = 0           # TODO: b * d_ta2
b_ta2 = 0           # TODO: a * d_ta2

assert (a_ta2, b_ta2) == (6.0, -4.0), f"a_ta2 = {a_ta2}, b_ta2 = {b_ta2}, lekin 6.0 va -4.0 chiqishi kerak"
print(a_ta2, b_ta2)     # 6.0 -4.0 chiqishi kerak

## 5. Mini-autograd: kompyuter buni o’zi qilsin

Yuqorida har qatorni **qo’lda** yozdik. Tarmoqda minglab amal bor, qo’lda mumkin emas. Shuning uchun kichik dastur yozamiz: **har son o’zi bilan ikki narsani eslab yuradi**:
- `data` — sonning qiymati;
- `grad` — xatoga ta’siri (orqaga o’tishda to’ldiriladi).

Har amal esa **qaysi sonlardan hosil bo’lganini** eslaydi va orqaga o’tishda o’z qoidasini ishlatadi (qo’shish — o’zgarishsiz uzatadi, ko’paytirish — o’zgacha songa ko’paytiradi). Bunday dastur **autograd** («avtomatik gradient») deyiladi. Bizniki juda kichik: **Value** klassi.

Qo’shishning qoidasi tayyor. **Ko’paytirishning qoidasini** (`_backward`) siz yozasiz: yuqorida `a_ta = b * d_ta` deb yozgan edik.

**Mustaqil ish 4.** `__mul__` ichidagi `_backward` ni to’ldiring. Qoida: `self.grad += other.data * out.grad` va `other.grad += self.data * out.grad`. («out.grad» — natijaning ta’siri.)

In [ ]:
class Value:
    def __init__(self, data, _children=()):
        self.data = data                       # sonning qiymati
        self.grad = 0.0                        # xatoga ta'siri (boshida 0)
        self._backward = lambda: None          # orqaga o'tishda nima qilish kerak
        self._prev = set(_children)            # qaysi sonlardan hosil bo'lgan

    def __add__(self, other):                  # qo'shish: a + b
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad += out.grad              # ta'sir o'zgarishsiz uzatiladi
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):                  # ko'paytirish: a * b
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():

            pass                               # TODO: self.grad += ...  va  other.grad += ...
        out._backward = _backward
        return out

    def tanh(self):                            # tanh aktivatsiyasi
        t = math.tanh(self.data)
        out = Value(t, (self,))
        def _backward():
            self.grad += (1 - t * t) * out.grad    # tanh ning qiyaligi (formulasi tayyor)
        out._backward = _backward
        return out

    def __pow__(self, n):                      # daraja: a ** n
        out = Value(self.data ** n, (self,))
        def _backward():
            self.grad += n * self.data ** (n - 1) * out.grad
        out._backward = _backward
        return out

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other

    def backward(self):                        # orqaga o'tish: oxiridan boshlab hammaga ta'sirni tarqatadi
        tartib, korilgan = [], set()
        def qur(v):
            if v not in korilgan:
                korilgan.add(v)
                for bola in v._prev:
                    qur(bola)
                tartib.append(v)
        qur(self)
        self.grad = 1.0                        # oxirgi sonning o'z ta'siri = 1
        for v in reversed(tartib):             # oxiridan boshlab
            v._backward()

Tekshiramiz: yuqoridagi **L = (a·b + c)·f** ni endi `Value` bilan hisoblaymiz. Ta’sirlar yana 6, −4, −2, 4 chiqishi kerak.

In [ ]:
a = Value(2.0); b = Value(-3.0); c = Value(10.0); f = Value(-2.0)

L = (a * b + c) * f        # hisoblash grafi o'zi qurildi
L.backward()               # orqaga o'tish: hammaning grad i to'ldirildi

print("L =", L.data)
print("a:", a.grad, " b:", b.grad, " c:", c.grad, " f:", f.grad)

assert (a.grad, b.grad, c.grad, f.grad) == (6.0, -4.0, -2.0, 4.0), "grad lar 6, -4, -2, 4 bo'lishi kerak. __mul__ ichidagi _backward ni to'ldirdingizmi?"

**Mos keldi!** `Value` klassi qo’lda yozgan hisobimizni o’zi qiladi. Siz yozgan **ko’paytirish qoidasi** (2 qator) butun tarmoqni o’qitishning yuragi.

## 6. `Value` dan tarmoq yasaymiz

Endi neyron va tarmoqni **`Value` sonlaridan** yig’amiz. Hammasi tayyor, faqat o’qing:

- **Neuron**: har kirish uchun `w` (boshida tasodifiy son), bitta `b`. Chaqirilganda: `w₁x₁ + w₂x₂ + … + b`, keyin aktivatsiya (tanh);
- **MLP** (ko’p qatlamli tarmoq): neyronlar qatorlari. `[2, 4, 1]` degani: **2 kirish → 4 neyronli yashirin qatlam → 1 chiqish**.

In [ ]:
class Neuron:
    def __init__(self, n, aktivatsiya):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(n)]     # har kirish uchun tasodifiy og'irlik
        self.b = Value(0.0)
        self.aktivatsiya = aktivatsiya

    def __call__(self, x):
        ball = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)    # w1*x1 + w2*x2 + ... + b
        return ball.tanh() if self.aktivatsiya else ball

    def parameters(self):
        return self.w + [self.b]


class MLP:
    def __init__(self, olchamlar):                 # masalan [2, 4, 1]
        self.qatlamlar = [
            [Neuron(olchamlar[i], i < len(olchamlar) - 2) for _ in range(olchamlar[i + 1])]
            for i in range(len(olchamlar) - 1)
        ]

    def __call__(self, x):
        for qatlam in self.qatlamlar:
            x = [neyron(x) for neyron in qatlam]
        return x[0]

    def parameters(self):
        return [p for qatlam in self.qatlamlar for n in qatlam for p in n.parameters()]

In [ ]:
random.seed(1)                     # har safar bir xil tasodifiy sonlar
tarmoq = MLP([2, 4, 1])            # 2 kirish -> 4 neyron -> 1 chiqish

print("og'irliklar soni:", len(tarmoq.parameters()))      # 2*4 + 4 + 4*1 + 1 = 17
print("o'qitilmagan tarmoq javobi:", [round(tarmoq(x).data, 2) for x in X])

Tarmoqda **17 ta son** (8 + 4 + 4 + 1). O’qitilmagan tarmoq taxminan tasodifiy javoblar beradi. Solishtirish uchun: Face ID tarmog’i — **millionlab** son, ChatGPT — **yuz milliardlab** son. Bizniki bolalar o’yinchog’i, lekin **qoida bir xil**.

## 7. Tarmoqni o’qitamiz

Hamma qismlar tayyor. O’qitish tsikli **5-darsdagi bilan bir xil**, faqat qiyalikni `Value` o’zi hisoblaydi:

1. **oldinga**: tarmoq to’rt holat uchun javob beradi;
2. **loss**: javoblar va to’g’ri javoblar farqining kvadrati o’rtachasi (5-darsdagi MSE);
3. **gradlarni nolga tushiramiz** (aks holda eskilari qo’shilib ketadi);
4. **orqaga**: `loss.backward()`: hamma og’irlikning ta’siri topiladi;
5. **qadam**: har og’irlikni qiyalikka **teskari** surish: `p.data = p.data - lr * p.grad`.

**Mustaqil ish 5.** Qadamni (5-bo’lim) yozing. Bu 5-darsdagi `k = k - lr * qiyalik` ning aynan o’zi.

In [ ]:
# TODO: 5-qadamni (og'irlikni yangilash) yozing
def o_qit(yashirin=4, lr=0.1, qadamlar=300, urug=1):
    random.seed(urug)
    model = MLP([2, yashirin, 1])
    tarix = []                                                  # loss tarixi

    for qadam in range(qadamlar):
        javoblar = [model(x) for x in X]                        # 1. oldinga
        loss = sum(((j - y) ** 2 for j, y in zip(javoblar, Y)), Value(0.0)) * (1 / 4)   # 2. loss

        for p in model.parameters():                            # 3. gradlarni nolga tushiramiz
            p.grad = 0.0
        loss.backward()                                         # 4. orqaga o'tish

        for p in model.parameters():                            # 5. qadam
            p.data = p.data                                     # TODO: p.data - lr * p.grad
        tarix.append(loss.data)
    return model, tarix


model, tarix = o_qit()
for qadam in [0, 50, 100, 150, 200, 299]:
    print(f"{qadam}-qadam: loss = {tarix[qadam]:.4f}")

assert tarix[-1] < 0.01, f"oxirgi loss = {tarix[-1]:.3f}, kamayishi kerak edi. p.data = p.data - lr * p.grad yozdingizmi?"

Loss **0.61 dan deyarli 0 gacha** tushdi. Tarmoq lampochka qoidasini o’zi o’rgandi, **hech kim og’irlikni qo’lda tanlamadi**. Javoblarini ko’ramiz va loss grafigini chizamiz.

In [ ]:
print("x1 x2 | tarmoq javobi | kerak edi")
for x, y in zip(X, Y):
    print(f" {x[0]}  {x[1]} |     {model(x).data:5.2f}     |   {y}")

plt.plot(tarix)
plt.xlabel("qadam"); plt.ylabel("loss")
plt.title("Tarmoq o'rganmoqda: loss kamayadi")
plt.show()

Javoblar: **0.00, 1.00, 1.00, 0.00**. Aynan lampochka jadvali. Loss grafigi 5-darsdagi grafikka o’xshaydi: avval tez tushadi, keyin tekislanadi.

**Muhim:** o’qitilgan tarmoq **bizning qo’lda yig’gan tarmoqdan boshqacha** og’irliklarni topdi (tanh bilan, boshqa sonlar). Lekin **vazifani bajaradi**. Tarmoq bir masalani turli yo’l bilan hal qila oladi.

**Sinab ko’ring.** Yashirin qatlamda **bitta** neyron qoldiring: `o_qit(yashirin=1)`. Avval taxmin qiling: o’rganadimi? Oxirgi loss nechaga yetadi?

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

Bitta yashirin neyron bilan loss **0.24 da qoldi**: tarmoq to’rt holatni ajrata olmadi. Neyronlar soni muhim. 3 va 4 ta neyron bilan loss deyarli 0 ga tushadi.

**Sinab ko’ring.** `o_qit(lr=0.001)` va `o_qit(lr=0.1)` ni solishtiring. 5-darsni eslang: juda kichik `lr` nima qiladi?

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

## 8. TensorFlow Playground: tarmoqni ko’rib o’ynaymiz

Brauzerda ochiq sayt: **https://playground.tensorflow.org**. U yerda tarmoqni sichqoncha bilan yig’ib, o’rganishini **jonli** ko’rasiz.

Vazifalar (har biri 1–2 daqiqa):
1. Chap tomondan **XOR**-ga o’xshash to’rtinchi ma’lumotni tanlang (to’rt burchakda aralash nuqtalar). **Yashirin qatlamlarni 0 ga** tushiring (`−` tugmasi). ▶ bosing: o’rganadimi?
2. **1 qatlam, 1 neyron** qo’ying. O’rganadimi? (Biz ko’rdik: yo’q.)
3. **1 qatlam, 3–4 neyron** qo’ying. Endi-chi?
4. Aktivatsiyani **Linear** ga almashtiring (aktivatsiyasiz). Qatlam va neyronlarni ko’paytiring. Baribir o’rganmaydi: nega? (3-bo’limdan eslang.)
5. Aktivatsiyani **ReLU** yoki **Tanh** ga qaytaring va neyronlar ustiga sichqonchani olib boring: har neyron **o’z «chizig’ini»** chizayotganini ko’rasiz. Tarmoq shu chiziqlarni birlashtiradi.

## 9. Siz buni har kuni ishlatasiz

Bugun yozgan tarmoq 17 sondan iborat. Xuddi shu qoida, faqat kattaroq:
- **Face ID**: telefonni ochayotganda yuzingizni tanish uchun millionlab sonli tarmoq;
- **TikTok va YouTube tavsiyasi**: sizga qaysi videoni ko’rsatishni tarmoq tanlaydi;
- **Tarjimon, ovozli yordamchi, klaviaturada keyingi so’zni taxmin qilish**;
- **ChatGPT**: yuz milliardlab son.

Hammasining o’qitilishi **bir xil 4 qadam**: **oldinga → loss → orqaga → qadam**. Bugun siz shu tsiklni noldan yozdingiz. Keyingi darsda (7-dars) **PyTorch** kutubxonasini ko’ramiz: u xuddi shu `Value` klassining katta, juda tez akasi. Bugun yozgan `backward()` o’rniga u bir qatorda ishlaydi.

## Uy ishi

**1. Yashirin neyronlar soni.** `o_qit` ni `yashirin = 1, 2, 3, 8` bilan ishga tushiring. Har birining loss grafigini **bitta rasmga** chizing va raqamlar bilan yozing: nechta neyrondan boshlab tarmoq XOR ni o’rgandi? Ko’p neyron doim yaxshimi?

**2. Qo’lda hisoblash grafi.** Formula: **L = (x · w + b − y)²**, sonlar: x = 2, w = 3, b = 1, y = 5. Qog’ozda hisoblash grafini chizing (har amal bitta qutu), har qutuga qiymatni yozing, so’ng **orqaga** hisoblab `w` va `b` ning ta’sirini (gradini) toping. Keyin `Value` bilan tekshiring.

**3. Qisqa yozma javob (2–3 gap).** Nega aktivatsiyasiz tarmoq (qancha qatlam bo’lsa ham) XOR ni o’rgana olmaydi?

Boshlash uchun shablon (1-vazifa):

In [ ]:
for yashirin in [1, 2, 3, 8]:
    m, t = o_qit(yashirin=yashirin)
    plt.plot(t, label=f"{yashirin} neyron")

plt.xlabel("qadam"); plt.ylabel("loss"); plt.legend()
plt.show()

**Javobingiz:** shu katakni ikki marta bosib yozing.

## Chuqurroq (ixtiyoriy): ReLU bilan o’qitish

Biz `Value` ga **tanh** qo’shdik. Siz **ReLU** ni qo’shing:
1. `Value` ichiga `relu` metodini yozing: `data = max(0, self.data)`; orqaga o’tishda ta’sir faqat `data > 0` bo’lsa uzatiladi: `self.grad += (out.data > 0) * out.grad`.
2. `Neuron.__call__` da `ball.tanh()` o’rniga `ball.relu()` ni qo’ying va `o_qit` ni `urug = 1…10` bilan ishga tushiring.

**Savol:** nechta urug’da tarmoq o’rgandi (oxirgi loss < 0.01)? Nega ReLU ba’zan «o’lib qoladi» (ba’zi neyronlar hech qachon signal bermaydi)? Maslahat: ball har doim manfiy bo’lsa, ReLU qiyaligi 0, demak o’rganish to’xtaydi.

## Yakun

Bugun siz:
- **neyronni** tushundingiz: og’irlikli yig’indi + aktivatsiya;
- bitta neyron lampochka (XOR) masalasini yecha olmasligini va **ikki qatlam** buni hal qilishini ko’rdingiz;
- **aktivatsiyasiz** tarmoq bitta chiziqqa aylanib ketishini bildingiz;
- **hisoblash grafi** va **backpropagation** (ta’sir zanjiri orqaga) nima ekanini o’zingiz hisoblab ko’rdingiz;
- o’zingizning **mini-autograd** (`Value`) dasturingizni yozdingiz va undan tarmoq o’qitdingiz.

Keyingi darsda (7-dars) shu tsiklni **PyTorch** da, haqiqiy kutubxonada, tez va katta ma’lumot bilan yozamiz.